In [1]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_DATASETS_CACHE"] = "D:/HuggingFaceCache"
os.environ["TRANSFORMERS_CACHE"] = "E:/seminar/HuggingFaceCache/models"
os.environ["HF_HOME"] = "E:/seminar/HuggingFaceCache"


from datasets import load_dataset, config

print("Using cache dir:", config.HF_DATASETS_CACHE)  

Using cache dir: D:\HuggingFaceCache


In [2]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
# from datasets import load_dataset, DatasetDict

# common_voice = DatasetDict()

# # common_voice["train"]= load_dataset("Minutor/Odia-data-collection", split="train")
# common_voice["test"]= load_dataset("Minutor/Odia-data-collection",  split="valid")


# from datasets import Audio

# common_voice= common_voice.cast_column("audio", Audio(sampling_rate=16000,decode=False))


Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

In [2]:
#zero shot
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict()

common_voice["train"]= load_dataset("cdactvm/odia_data_v1", split="train")
# common_voice["test"]= load_dataset("Minutor/Odia-data-collection",  split="valid")


from datasets import Audio

common_voice= common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
# keep_cols = [ "audio_filepath", "text"]

# # Drop the rest
# common_voice = DatasetDict({
#     split: ds.remove_columns([col for col in ds.column_names if col not in keep_cols])
#     for split, ds in common_voice.items()
# })

In [ ]:
# common_voice['train'][0]

{'audio': {'path': 'Regional-Cuttack-Odia-1430-2020111115753_sent_50.mp3',
  'array': array([ 0.02846138,  0.04058887,  0.04585309, ..., -0.00411778,
         -0.00489142, -0.00391264], shape=(102400,)),
  'sampling_rate': 16000},
 'sentence': 'ପ୍ରଧାନମନ୍ତ୍ରୀ କହିଛନ୍ତି ଔଷଧ ନ ଆସିବା ପର୍ଯ୍ୟନ୍ତ କୋଭିଡ୍ ନିୟମ ପାଳନରେ କୌଣସି କୋହଳ ମନୋଭାବ ପୋଷଣ କରିବା ଠିକ୍ ନୁହେଁ'}

In [4]:
import torch
torch.cuda.empty_cache()
#inference

import torch
from datasets import load_dataset
from transformers import AutoProcessor,  AutoModelForSpeechSeq2Seq, BitsAndBytesConfig

model_name="openai/whisper-tiny"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(model_name, language="Bengali", task="transcribe")
model =  AutoModelForSpeechSeq2Seq.from_pretrained(
    model_name,
    # torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
    attn_implementation="sdpa"
)
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)


e:\seminar\whiper\lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
e:\seminar\whiper\lib\site-packages\accelerate\utils\modeling.py:821: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


In [5]:
from jiwer import wer, cer
from tqdm import tqdm
import torch
from torch.cuda.amp import autocast

# Normalization helper from Whisper tokenizer
normalize = tokenizer._normalize  # assumes you're using WhisperTokenizer

cer_avg = []
wer_avg = []

model.eval()
model = model.to("cuda")

for i in tqdm(range(100), desc="Evaluating Whisper"):
    sample = common_voice['train'][i]
    audio_array = sample['audio']["array"]
    sampling_rate = sample['audio']["sampling_rate"]
    ground_truth = sample["sentence"]

    # Step 1: Feature extraction
    input_features = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to("cuda")

    # Step 2: Inference
    with torch.no_grad():
        with autocast():
            predicted_ids = model.generate(input_features)

    # Step 3: Decode
    transcription = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Step 4: Normalize both
    normalized_pred = normalize(transcription)
    normalized_gt = normalize(ground_truth)

    # Step 5: Compute metrics
    wer_score = wer(normalized_gt, normalized_pred)
    cer_score = cer(normalized_gt, normalized_pred)

    wer_avg.append(wer_score)
    cer_avg.append(cer_score)


Evaluating Whisper:   0%|          | 0/100 [00:00<?, ?it/s]C:\Users\subha\AppData\Local\Temp\ipykernel_15268\4263454484.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable res

In [6]:
# Final average
print(f"\nAverage WER over {len(wer_avg)} samples: {100 * sum(wer_avg)/len(wer_avg):.2f}%")
print(f"Average CER over {len(cer_avg)} samples: {100 * sum(cer_avg)/len(cer_avg):.2f}%")


Average WER over 100 samples: 203.67%
Average CER over 100 samples: 260.45%


In [7]:
from jiwer import wer, cer
from tqdm import tqdm
import torch
from torch.cuda.amp import autocast  # optional, for fp16 inference

cer_avg = []
wer_avg = []

model.eval()
model = model.to("cuda")

for i in tqdm(range(int(len(common_voice['train']['sentence'])/3)), desc="Evaluating Whisper"):
    sample = common_voice['train'][i]
    audio_array = sample['audio']["array"]
    sampling_rate = sample['audio']["sampling_rate"]
    ground_truth = sample["sentence"]

    # Step 1: Extract input features (always float32)
    input_features = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to("cuda")

    # Step 2: Generate prediction (safe and compatible)
    with torch.no_grad():
        # Optional: use autocast if model is in fp16
        with autocast():
            predicted_ids = model.generate(input_features)

    # Step 3: Decode prediction
    transcription = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Step 4: Compute WER & CER
    wer_score = wer(ground_truth, transcription)
    cer_score = cer(ground_truth, transcription)

    wer_avg.append(wer_score)
    cer_avg.append(cer_score)

    # print(f"\n[{i+1}]")
    # print("Predicted:", transcription)
    # print("Reference:", ground_truth)
    # print(f"WER: {wer_score:.3f} | CER: {cer_score:.3f}")




Evaluating Whisper:   0%|          | 0/17744 [00:00<?, ?it/s]C:\Users\subha\AppData\Local\Temp\ipykernel_2776\614837246.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Evaluating Whisper: 100%|██████████| 17744/17744 [5:22:04<00:00,  1.09s/it]  


In [ ]:
jk

In [9]:
# Final average
print(f"\nAverage WER over {len(wer_avg)} samples: {100 * sum(wer_avg)/len(wer_avg):.2f}%")
print(f"Average CER over {len(cer_avg)} samples: {100 * sum(cer_avg)/len(cer_avg):.2f}%")


Average WER over 17744 samples: 269.15%
Average CER over 17744 samples: 222.68%


In [9]:
encoded=tokenizer.encode(common_voice['test'][0]['text'])
decoded=tokenizer.decode(encoded)
decoded

'<|startoftranscript|><|bn|><|transcribe|><|notimestamps|>ପାଞ୍ଚେ ଅଗଷ୍ଟ ଦୁଇହଜାର ପାଞ୍ଚେ ବାର ମାର୍ଚ୍ଚ ଦୁଇହଜାରବାର ତେର ମେ ଦୁଇହଜାର ସତର ଚବିଶି ସେପ୍ଟେମ୍ବର ଦୁଇହଜାର ଏଗାର ଦଶ ଜାନୁୟାରୀ ଦୁଇହଜାରଦୁଇ<|endoftext|>'

In [10]:
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model

config = LoraConfig(r=512, lora_alpha=1024, target_modules=["q_proj", "v_proj", "q_proj", "out_proj"], lora_dropout=0.05, bias="none")

model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 14,155,776 || all params: 51,916,416 || trainable%: 27.2665


In [11]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(model_name, language="Bengali", task="transcribe")
tokenizer = processor.tokenizer
max_length = 200  # Max tokens for the text

def prepare_dataset(batch):
    try:
        # Truncate the text if it exceeds max_length
        encoded = tokenizer(batch["text"], truncation=True, max_length=max_length)
        truncated_text = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)

        # Process audio and truncated text
        inputs = processor(
            batch["audio_filepath"]["array"],
            sampling_rate=batch["audio_filepath"]["sampling_rate"],
            text=truncated_text,
            truncation=True,
            return_attention_mask=True,
            return_tensors=None
        )

        return {
            "input_features": inputs["input_features"][0],
            "labels": inputs["labels"],
            "attention_mask": inputs["attention_mask"]
        }

    except Exception as e:
        print(f"Skipped sample due to error: {e}")
        return None



common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"])


Map:   0%|          | 0/230514 [00:00<?, ? examples/s]

Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C54DB330>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C3A22D90>: Format not recognised.
Skipped sample due to error: Error opening <_io.BytesIO object at 0x00000204C3A22D90>: Format not recognised.
Skipped sa

Map:   0%|          | 0/3303 [00:00<?, ? examples/s]

In [12]:
common_voice

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels', 'attention_mask'],
        num_rows: 230493
    })
    test: Dataset({
        features: ['input_features', 'labels', 'attention_mask'],
        num_rows: 3303
    })
})

In [13]:
model.generation_config.language = "Bengali"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = tokenizer.get_decoder_prompt_ids(
    language="Bengali", task="transcribe"
)

In [14]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [15]:

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [16]:
import evaluate

metric = evaluate.load("wer")

In [17]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [33]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./lora-whisper",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=1,
    warmup_steps=600,
    gradient_checkpointing=True,
    fp16=True,
    save_steps=1000,
    logging_steps=1000,
    eval_strategy="no",  # <-- disables all evaluation
    predict_with_generate=False,
    generation_max_length=200,
    load_best_model_at_end=False,  # <-- this must also be False since no eval is happening
    report_to=["tensorboard"],
    label_names=["labels"],
)


In [34]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,  # your PEFT-wrapped model
    args=training_args,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    
)


C:\Users\subha\AppData\Local\Temp\ipykernel_9344\2650834597.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [35]:
processor.save_pretrained(training_args.output_dir)

[]

In [36]:
import gc
gc.collect()


6383

In [37]:
torch.cuda.empty_cache()
model.config.use_cache = False

model.gradient_checkpointing_disable()

model.train()
trainer.train()

Step,Training Loss
1000,0.683600
2000,0.280900
3000,0.240400
4000,0.219800
5000,0.205400
6000,0.196500
7000,0.188100
8000,0.182100
9000,0.175900
10000,0.171400


e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\site-packages\torch\utils\checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
e:\seminar\whiper\lib\

TrainOutput(global_step=14406, training_loss=0.22553659746518784, metrics={'train_runtime': 55601.9205, 'train_samples_per_second': 4.145, 'train_steps_per_second': 0.259, 'total_flos': 1.037292316766208e+19, 'train_loss': 0.22553659746518784, 'epoch': 1.0})

In [ ]:
# model.save_pretrained('lora_model_after_training')

In [15]:
torch.cuda.empty_cache()

In [3]:
from peft import PeftModel
from transformers import WhisperForConditionalGeneration
model_name="openai/whisper-tiny"
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model = PeftModel.from_pretrained(model,"lora_model_after_training")
model = model.merge_and_unload()

e:\seminar\whiper\lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
e:\seminar\whiper\lib\site-packages\safetensors\torch.py:315: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  result[k] = f.get_tensor(k)


In [4]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(model_name, language="Bengali", task="transcribe")

In [ ]:
# from datasets import load_dataset, DatasetDict

# common_voice = DatasetDict()

# # common_voice["train"]= load_dataset("Minutor/Odia-data-collection", split="train")
# common_voice["test"]= load_dataset("Minutor/Odia-data-collection",  split="valid")


# from datasets import Audio

# common_voice= common_voice.cast_column("audio", Audio(sampling_rate=16000,decode=False))

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

In [5]:
common_voice

DatasetDict({
    test: Dataset({
        features: ['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized', '__index_level_0__', 'audio'],
        num_rows: 3303
    })
})

In [7]:
from jiwer import wer, cer
from tqdm import tqdm
import torch
from torch.cuda.amp import autocast

# Normalization helper from Whisper tokenizer
normalize = tokenizer._normalize  # assumes you're using WhisperTokenizer

cer_avg = []
wer_avg = []

model.eval()
model = model.to("cuda")

for i in tqdm(range(int(len(common_voice['train']['sentence'])/3)), desc="Evaluating Whisper"):
    sample = common_voice['train'][i]
    audio_array = sample['audio']["array"]
    sampling_rate = sample['audio']["sampling_rate"]
    ground_truth = sample["sentence"]

    # Step 1: Feature extraction
    input_features = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to("cuda")

    # Step 2: Inference
    with torch.no_grad():
        with autocast():
            predicted_ids = model.generate(input_features)

    # Step 3: Decode
    transcription = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Step 4: Normalize both
    normalized_pred = normalize(transcription)
    normalized_gt = normalize(ground_truth)

    # Step 5: Compute metrics
    wer_score = wer(normalized_gt, normalized_pred)
    cer_score = cer(normalized_gt, normalized_pred)

    wer_avg.append(wer_score)
    cer_avg.append(cer_score)


Evaluating Whisper:   0%|          | 0/17744 [00:00<?, ?it/s]C:\Users\subha\AppData\Local\Temp\ipykernel_15732\1755602376.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Evaluating Whisper: 100%|██████████| 17744/17744 [7:32:30<00:00,  1.53s/it]  


In [8]:
# Final average
print(f"\nAverage WER over {len(wer_avg)} samples: {100 * sum(wer_avg)/len(wer_avg):.2f}%")
print(f"Average CER over {len(cer_avg)} samples: {100 * sum(cer_avg)/len(cer_avg):.2f}%")


Average WER over 17744 samples: 66.66%
Average CER over 17744 samples: 45.08%


In [8]:
from jiwer import wer, cer
from tqdm import tqdm
import torch
from torch.cuda.amp import autocast  # optional, for fp16 inference

cer_avg = []
wer_avg = []

model.eval()
model = model.to("cuda")

for i in tqdm(range(int(len(common_voice['train']['sentence'])/3)), desc="Evaluating Whisper"):
    sample = common_voice['train'][i]
    audio_array = sample['audio']["array"]
    sampling_rate = sample['audio']["sampling_rate"]
    ground_truth = sample["sentence"]

    # Step 1: Extract input features (always float32)
    input_features = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to("cuda")

    # Step 2: Generate prediction (safe and compatible)
    with torch.no_grad():
        # Optional: use autocast if model is in fp16
        with autocast():
            predicted_ids = model.generate(input_features)

    # Step 3: Decode prediction
    transcription = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Step 4: Compute WER & CER
    wer_score = wer(ground_truth, transcription)
    cer_score = cer(ground_truth, transcription)

    wer_avg.append(wer_score)
    cer_avg.append(cer_score)

    # print(f"\n[{i+1}]")
    # print("Predicted:", transcription)
    # print("Reference:", ground_truth)
    # print(f"WER: {wer_score:.3f} | CER: {cer_score:.3f}")




Evaluating Whisper:   0%|          | 0/17744 [00:00<?, ?it/s]C:\Users\subha\AppData\Local\Temp\ipykernel_8884\614837246.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Evaluating Whisper: 100%|██████████| 17744/17744 [7:08:10<00:00,  1.45s/it]  


In [9]:
print(f"\nAverage WER over {len(wer_avg)} samples: {100 * sum(wer_avg)/len(wer_avg):.2f}%")
print(f"Average CER over {len(cer_avg)} samples: {100 * sum(cer_avg)/len(cer_avg):.2f}%")


Average WER over 17744 samples: 78.64%
Average CER over 17744 samples: 48.70%
